<a href="https://colab.research.google.com/github/Dmitze/Dmitze/blob/main/final_project_Dmytro_Shyvachov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Імпорт бібліотек

In [ ]:
!pip install catboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, TargetEncoder, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, VotingClassifier, RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectFromModel, SelectKBest, mutual_info_classif
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from google.colab import drive

import warnings
warnings.filterwarnings('ignore')

# 2. Завантаження та базова обробка

In [ ]:
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Colab Notebooks/kaggle/'
train = pd.read_csv(base_path + 'final_proj_data.csv')
test = pd.read_csv(base_path + 'final_proj_test.csv')
sample_sub = pd.read_csv(base_path + 'final_proj_sample_submission.csv')

X = train.drop('y', axis=1)
y = train['y']

X['missing_count'] = X.isnull().sum(axis=1)
test['missing_count'] = test.isnull().sum(axis=1)

cat_cols_raw = X.select_dtypes(include=['object']).columns.tolist()
X = X.astype({col: str for col in cat_cols_raw})
test = test.astype({col: str for col in cat_cols_raw})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 3. Підготовка даних (KMeansFeaturizer)

In [ ]:
class KMeansFeaturizer(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=5, random_state=42):
        self.n_clusters = n_clusters
        self.random_state = random_state
    def fit(self, X, y=None):
        self.imputer_ = SimpleImputer(strategy='median').fit(X)
        self.scaler_ = StandardScaler().fit(self.imputer_.transform(X))
        self.kmeans_ = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.kmeans_.fit(self.scaler_.transform(self.imputer_.transform(X)))
        return self
    def transform(self, X):
        return self.kmeans_.predict(
            self.scaler_.transform(self.imputer_.transform(X))
        ).reshape(-1, 1).astype(str)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
    ('power', PowerTransformer(method='yeo-johnson'))
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', TargetEncoder(target_type='binary', smooth='auto'))
])

kmeans_pipe = Pipeline(steps=[
    ('kmeans', KMeansFeaturizer(n_clusters=5, random_state=42)),
    ('encoder', TargetEncoder(target_type='binary', smooth='auto'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols),
        ('kmeans_feat', kmeans_pipe, num_cols)
    ]
)

# 4. Ансамбль ("Балансування через ваги", "Відбір ознак")

In [ ]:
# згідно з абляцією, SMOTE в дуже багатовимірному просторі генерує шум.
# вбудовані ваги (class_weight='balanced')
feature_selection = SelectFromModel(
    RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced'),
    threshold='median' # відкидаємо 50% найгірших шумових ознак
)

hgb = HistGradientBoostingClassifier(
    random_state=42, max_iter=300, learning_rate=0.05, l2_regularization=0.5,
    class_weight='balanced'
)

cat = CatBoostClassifier(
    random_state=42, iterations=300, learning_rate=0.05,
    thread_count=1, verbose=False,
    auto_class_weights='Balanced'
)

lgbm = LGBMClassifier(
    random_state=42, n_estimators=300, learning_rate=0.05,
    n_jobs=1, verbosity=-1,
    class_weight='balanced'
)

voting = VotingClassifier(estimators=[
    ('hgb', hgb),
    ('cat', cat),
    ('lgbm', lgbm)
], voting='soft')

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selection', feature_selection),
    ('classifier', voting)
])

# 5. Чесний пошук порогу (Multi-seed CV)

In [ ]:
all_thresholds = []
all_oof_scores = []

for seed in [42, 123, 2024]:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    proba = cross_val_predict(
        final_pipeline, X, y, cv=cv, method='predict_proba', n_jobs=1
    )[:, 1]

    thresholds = np.linspace(0.01, 0.99, 200)
    best_t, best_s = 0.5, 0
    for t in thresholds:
        s = balanced_accuracy_score(y, (proba >= t).astype(int))
        if s > best_s:
            best_s, best_t = s, t

    all_thresholds.append(best_t)
    all_oof_scores.append(best_s)
    print(f"Seed={seed}: Balanced_Accuracy={best_s:.4f}, Threshold={best_t:.4f}")

final_thresh = np.mean(all_thresholds)
print(f"\nсередній поріг: {final_thresh:.4f} (std={np.std(all_thresholds):.4f})")
print(f"середній OOF score: {np.mean(all_oof_scores):.4f} (std={np.std(all_oof_scores):.4f})")

Seed=42: Balanced_Accuracy=0.8976, Threshold=0.2661
Seed=123: Balanced_Accuracy=0.9031, Threshold=0.2562
Seed=2024: Balanced_Accuracy=0.8944, Threshold=0.2365

середній поріг: 0.2529 (std=0.0123)
середній OOF score: 0.8984 (std=0.0036)


# 6. Фінальне навчання (Full Train) & Статистика

In [ ]:
print("обучаю фінальну модель на всіх тренувальних даних...")
final_pipeline.fit(X, y)

# перевіряємо скільки ознак пройшло відбір
fs = final_pipeline.named_steps['feature_selection']
print(f"\nВідібрано ознак: {fs.get_support().sum()} з {fs.n_features_in_}")

print("формую тестові передбачення")
test_proba = final_pipeline.predict_proba(test)[:, 1]
final_preds = (test_proba >= final_thresh).astype(int)

print("\n--- ДІАГНОСТИКА ---")
print(f"Доля класу 1 у train: {y.mean():.4f}")
print(f"Доля класу 1 у test: {final_preds.mean():.4f}")

submission = pd.DataFrame({
    'index': range(len(final_preds)),
    'y': final_preds
})
submission.to_csv('submission.csv', index=False)
print("\nфайл submission.csv успішно згенеровано")

обучаю фінальну модель на всіх тренувальних даних...

Відібрано ознак: 202 з 403
формую тестові передбачення

--- ДІАГНОСТИКА ---
Доля класу 1 у train: 0.1305
Доля класу 1 у test: 0.2360

файл submission.csv успішно згенеровано


## 7. Висновки

Протягом роботи над проєктом перепробував купу варіантів. Пропуски і дисбаланс класів добряче попили крові, але в результаті вийшло непогано.

**Що в підсумку дало результат:**
1. **Фічі та кластеризація:** Замість того щоб просто дропати пропуски, додав колонку missing_count — це виявився класний сигнал. Також написав свій KMeansFeaturizer для кластеризації. Головне було засунути туди скейлер, щоб не зловити data leakage на крос-валідації.
2. **Відмова від SMOTE:** Довго мучився зі SMOTETomek, але на 200+ ознаках він тільки генерував шум. Як тільки я його викинув і просто додав class_weight='balanced' у самі моделі — скор сильно зріс.
3. **Відбір ознак:** Щоб бустинги не перенавчались на смітті, додав SelectFromModel з RandomForest. Він відкинув майже 50% слабких ознак.
4. **Ансамбль:** Зібрав VotingClassifier на 3-х моделях "CatBoost", "LightGBM" та "HistGradientBoosting"." Слабкі сторони однієї моделі перекриваються іншими.
5. **Усереднення порогу:** Стандартний поріг 0.5 на дисбалансі не працює. Замість одного спліта я зробив чесний OOF (cross_val_predict) і перевірив поріг на 3-х різних сідах (42, 123, 2024). Тепер поріг максимально стабільний.

Скор вийшов сильний, модель майже не оверфітить.